<div style="
    font-family: 'Trebuchet MS'; 
    padding: 30px; 
    border-radius: 60px; 
    background: linear-gradient(135deg, rgba(33,87,62,1), rgb(92,152,255));
    color: white;
    text-align: center;
    box-shadow: 0 8px 22px rgba(0,0,0,0.25);
">
    <h1 style="font-family: Trebuchet MS; padding: 12px; font-size: 48px; color:rgba(33, 87, 62, 1); text-align: center; line-height: 1.25;">
    <b>⚽In Match<span style="color: #000000"> Notebook 🎮📉</span></b><br>
  <span style="color: #000000; font-size: 24px">features contained :</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide suggested replacements for the players in the field during match ✨</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide suggested tactical style for the players in the field based on the real-time statistics from the match ✨</span>
</h1>

</div>


In [1]:
api_base = r'https://football-backend-app.victoriouswater-69fff737.swedencentral.azurecontainerapps.io/'

In [2]:
import numpy as np , json , requests
import pandas as pd
from google import genai
from pandas import DataFrame , Series

In [13]:
import requests
from requests.exceptions import JSONDecodeError, RequestException

def safe_get_json(url, default=None, timeout=10):
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()  # HTTP errors (4xx, 5xx)

        # Empty response
        if not r.text.strip():
            return default

        return r.json()

    except (JSONDecodeError, ValueError):
        # Response was not JSON
        return default

    except RequestException:
        # Network / timeout / HTTP error
        return default

In [14]:
def data():
    players_stats = []
    players_heatmaps = []
    players_shotmaps = []
    players_rating_breakdowns = []

    event_id = 13980104

    event_stats = safe_get_json(
        f'{api_base}/events/{event_id}/statistics',
        default={}
    )

    lineups = safe_get_json(
        f'{api_base}/events/{event_id}/lineups',
        default={"home": {"players": []}, "away": {"players": []}}
    )

    for player in lineups['home']['players'] + lineups['away']['players']:
        player_id = player['player']['id']

        players_stats.append(
            safe_get_json(
                f'{api_base}/events/{event_id}/player/{player_id}/statistics',
                default={}
            )
        )

        players_heatmaps.append(
            safe_get_json(
                f'{api_base}/events/{event_id}/player/{player_id}/heatmap',
                default=[]
            )
        )

        players_shotmaps.append(
            safe_get_json(
                f'{api_base}/events/{event_id}/player/{player_id}/shotmap',
                default=[]
            )
        )

        players_rating_breakdowns.append(
            safe_get_json(
                f'{api_base}/events/{event_id}/player/{player_id}/rating-breakdown',
                default={}
            )
        )

    return (
        event_stats,
        lineups,
        players_stats,
        players_heatmaps,
        players_shotmaps,
        players_rating_breakdowns
    )

In [ ]:
event_stats , lineups,  players_stats , players_heatmaps , players_shotmaps , players_rating_breakdowns = data()

In [28]:
import asyncio
import aiohttp

api_base = "YOUR_API_BASE"
event_id = 13980104

# limit concurrency (important for API stability)
semaphore = asyncio.Semaphore(10)


# ---------- safe async fetch ----------
async def fetch_json(session, url):
    async with semaphore:
        try:
            async with session.get(url) as response:
                # handle non-json responses safely
                if response.content_type != 'application/json':
                    text = await response.text()
                    return None if text.strip() == "" else text

                return await response.json()

        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None


# ---------- fetch player data ----------
async def fetch_player_data(session, player_id):
    stats_url = f'{api_base}/events/{event_id}/player/{player_id}/statistics'
    heatmap_url = f'{api_base}/events/{event_id}/player/{player_id}/heatmap'
    shotmap_url = f'{api_base}/events/{event_id}/player/{player_id}/shotmap'
    rating_url = f'{api_base}/events/{event_id}/player/{player_id}/rating-breakdown'

    stats, heatmap, shotmap, rating = await asyncio.gather(
        fetch_json(session, stats_url),
        fetch_json(session, heatmap_url),
        fetch_json(session, shotmap_url),
        fetch_json(session, rating_url),
        return_exceptions=True
    )

    return stats, heatmap, shotmap, rating


# ---------- main async function ----------
async def data_async():
    async with aiohttp.ClientSession() as session:

        # fetch event data in parallel
        event_stats, lineups = await asyncio.gather(
            fetch_json(session, f'{api_base}/events/{event_id}/statistics'),
            fetch_json(session, f'{api_base}/events/{event_id}/lineups')
        )

        players = lineups['home']['players'] + lineups['away']['players']

        # create tasks
        tasks = [
            fetch_player_data(session, player['player']['id'])
            for player in players
        ]

        results = await asyncio.gather(*tasks, return_exceptions=True)

        # containers
        players_stats = []
        players_heatmaps = []
        players_shotmaps = []
        players_rating_breakdowns = []

        # unpack safely
        for result in results:
            if isinstance(result, Exception):
                print("Task failed:", result)
                continue

            stats, heatmap, shotmap, rating = result

            players_stats.append(stats)
            players_heatmaps.append(heatmap)
            players_shotmaps.append(shotmap)
            players_rating_breakdowns.append(rating)

        return (
            event_stats,
            lineups,
            players_stats,
            players_heatmaps,
            players_shotmaps,
            players_rating_breakdowns
        )


# ---------- usage in Jupyter ----------
# IMPORTANT: use await (NOT asyncio.run)
# result = await data_async()

In [29]:
result = await data_async()

event_stats, lineups, players_stats, players_heatmaps, players_shotmaps, players_rating_breakdowns = result

Error fetching YOUR_API_BASE/events/13980104/statistics: YOUR_API_BASE/events/13980104/statistics
Error fetching YOUR_API_BASE/events/13980104/lineups: YOUR_API_BASE/events/13980104/lineups


TypeError: 'NoneType' object is not subscriptable

In [22]:

with open("event_stats2.json", "w", encoding="utf-8") as f:
    json.dump(event_stats, f, indent=2, ensure_ascii=False)


with open("lineups2.json", "w", encoding="utf-8") as f:
    json.dump(lineups, f, indent=2, ensure_ascii=False)


with open("players_stats2.json", "w", encoding="utf-8") as f:
    json.dump(players_stats, f, indent=2, ensure_ascii=False)


with open("players_shotmaps2.json", "w", encoding="utf-8") as f:
    json.dump(players_shotmaps, f, indent=2, ensure_ascii=False)


with open("players_heatmaps2.json", "w", encoding="utf-8") as f:
    json.dump(players_heatmaps, f, indent=2, ensure_ascii=False)


with open("players_rating_breakdowns2.json", "w", encoding="utf-8") as f:
    json.dump(players_rating_breakdowns, f, indent=2, ensure_ascii=False)


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">1. | Rolling Snapshot & Delta Engine </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">2. | Feature Engineering & Baseline Comparisons  </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">3. | Performance Deviation (Z-Score Model)   </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">4. | Substitution Urgency XGBoost   </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">5. | Pitch Grid Zone Threat Regressor    </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">6. | Formation Effectiveness Model     </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">7. | Inference Pipeline & LLM Aggregation    </div>